# Preprocessing and deterministic passage split

Extracted from the recovered Colab notebook. Original cleaning/split code and summary outputs are retained below; paths are adapted to the repository. Run from the repository root. The input is an authorized existing `data/clean_parallel_corpus.csv`, not a fresh upstream retrieval. Original source revision, subset selection, and corpus deduplication are unresolved. The final verification section was added on 2026-10-02.

Export safety was added on 2026-10-02: existing matching CSVs are preserved byte-for-byte; mismatched existing files cause an error.

In [ ]:
from pathlib import Path
import os
PROJECT_PATH = str(Path.cwd() / "data")
if not (Path(PROJECT_PATH) / "clean_parallel_corpus.csv").exists():
    raise FileNotFoundError("Place the authorized clean_parallel_corpus.csv in the repository data/ directory.")

def export_preserving_frozen(frame, path):
    import pandas as pd
    path = Path(path)
    if path.exists():
        existing = pd.read_csv(path)
        if not frame.reset_index(drop=True).equals(existing.reset_index(drop=True)):
            raise ValueError(f"Existing export differs; refusing to overwrite {path}")
        print(f"Preserved existing CSV bytes: {path}")
    else:
        frame.to_csv(path, index=False)

In [ ]:
import pandas as pd
import gc
import os

# Ensure PROJECT_PATH is defined (from cell 3ccd41ed)
# PROJECT_PATH is configured above
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv') # Define path for saved data

# --- Load from saved CSV only (no Hugging Face streaming fallback) ---
if os.path.exists(CLEAN_DATA_PATH):
    parallel_df = pd.read_csv(CLEAN_DATA_PATH)
    print(f"Data loaded from save-point: {len(parallel_df)} aligned rows from {CLEAN_DATA_PATH}.")
    # In this case, SEARCH_BUFFER represents the length of the loaded, already cleaned data
    SEARCH_BUFFER = len(parallel_df)
else:
    raise FileNotFoundError(f"No save-point found at {CLEAN_DATA_PATH}. Please ensure it has been created by running previous phases.")

display(parallel_df.head())

In [ ]:
import re
import gc # Import gc for explicit garbage collection

def clean_text(text):
    if not isinstance(text, str): return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Character repetition normalization (e.g., haaaaa -> haa)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Basic punctuation cleaning
    text = re.sub(r'[!?,\-\n]', ' ', text)
    return text.strip()

def contains_arabic(text):
    return bool(re.search('[\u0600-\u06FF]', str(text)))

# Safety check: only proceed if we have data
if not parallel_df.empty and 'text_az' in parallel_df.columns:
    print(f"Initial rows before cleaning: {len(parallel_df)}")

    # Apply cleaning
    parallel_df['text_az'] = parallel_df['text_az'].apply(clean_text)
    parallel_df['text_ar'] = parallel_df['text_ar'].apply(clean_text)
    gc.collect() # Free memory after creating new series

    # Discard rows where Arabizi column actually contains Arabic script
    initial_rows_after_cleaning = len(parallel_df)
    parallel_df = parallel_df[~parallel_df['text_az'].apply(contains_arabic)]
    print(f"Rows after filtering Arabic in Arabizi: {len(parallel_df)} (Removed {initial_rows_after_cleaning - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Word counts for length-ratio filtering
    # Create these columns
    parallel_df['len_az'] = parallel_df['text_az'].str.split().str.len()
    parallel_df['len_ar'] = parallel_df['text_ar'].str.split().str.len()
    gc.collect() # Free memory after creating new series

    # Avoid division by zero and perform initial length filter
    initial_rows_before_zero_filter = len(parallel_df)
    parallel_df = parallel_df[(parallel_df['len_az'] > 0) & (parallel_df['len_ar'] > 0)]
    print(f"Rows after removing zero-length sentences: {len(parallel_df)} (Removed {initial_rows_before_zero_filter - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Length-Ratio Filter: Discard if word count difference is > 2x
    parallel_df['ratio'] = parallel_df['len_az'] / parallel_df['len_ar']
    gc.collect() # Free memory after creating new series

    initial_rows_before_ratio_filter = len(parallel_df)
    parallel_df = parallel_df[(parallel_df['ratio'] >= 0.5) & (parallel_df['ratio'] <= 2.0)]
    print(f"Rows after length-ratio filter: {len(parallel_df)} (Removed {initial_rows_before_ratio_filter - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Explicitly drop temporary columns
    parallel_df = parallel_df.drop(columns=['len_az', 'len_ar', 'ratio'], errors='ignore')
    gc.collect() # Free memory after dropping columns

    print(f"Dataset cleaned. Remaining rows: {len(parallel_df)}")
    display(parallel_df.head())
else:
    print("Warning: parallel_df is empty or missing columns. Cannot perform cleaning.")

Initial rows before cleaning: 8045
Rows after filtering Arabic in Arabizi: 8045 (Removed 0)
Rows after removing zero-length sentences: 8045 (Removed 0)
Rows after length-ratio filter: 8045 (Removed 0)
Dataset cleaned. Remaining rows: 8045


In [ ]:
# Step 2: Create 'Save-Point' for the parallel corpus
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')

# Save to Drive
export_preserving_frozen(parallel_df, CLEAN_DATA_PATH)
print(f'Persistence Success: Cleaned data saved to {CLEAN_DATA_PATH}')

In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd

# Ensure PROJECT_PATH is defined
if 'PROJECT_PATH' not in globals():
    # PROJECT_PATH is configured above

# Ensure parallel_df is loaded if not already in memory
if 'parallel_df' not in globals():
    print("parallel_df not found in RAM. Attempting to load from save-point...")
    CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')
    if os.path.exists(CLEAN_DATA_PATH):
        parallel_df = pd.read_csv(CLEAN_DATA_PATH)
        print(f'Data loaded: {len(parallel_df)} rows from {CLEAN_DATA_PATH}.')
    else:
        raise FileNotFoundError(f"Clean parallel corpus not found at {CLEAN_DATA_PATH}. Please run Phase 2 & 3 first to create it.")

# Splitting the larger parallel_df
# First split: 80% Train, 20% Temporary (Val + Test)
train_df, temp_df = train_test_split(parallel_df, test_size=0.2, random_state=42)

# Second split: Split the 20% into half (10% Val, 10% Test)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print row counts
print(f"Total cleaned rows: {len(parallel_df)}")
print(f"--- Splits ---")
print(f"Train set:      {len(train_df)} rows")
print(f"Validation set: {len(val_df)} rows")
print(f"Test set:       {len(test_df)} rows")

# Save to CSV for the next algorithms within the PROJECT_PATH
export_preserving_frozen(train_df, os.path.join(PROJECT_PATH, 'toy_train.csv'))
export_preserving_frozen(val_df, os.path.join(PROJECT_PATH, 'toy_val.csv'))
export_preserving_frozen(test_df, os.path.join(PROJECT_PATH, 'toy_test.csv'))

print("\nSuccess: Scaled datasets split and saved to PROJECT_PATH.")

Total cleaned rows: 8045
--- Splits ---
Train set:      6436 rows
Validation set: 804 rows
Test set:       805 rows

Success: Scaled datasets split and saved to PROJECT_PATH.


## Later verification additions (2026-10-02)

The command below checks exact duplicates, ID assignments, and equality with the frozen CSV contents. It does not certify source-document independence or the history of test-informed development. File hashes and decoded content equality are reported separately. Outputs were reconstructed independently, not inherited from the original Colab run.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "evaluation.verify_preprocessing", "--corpus", "data/clean_parallel_corpus.csv", "--splits", "data", "--output", "results/local_split_verification.json"], check=True)